# product-taxonomy-bench: reproducible baselines

This notebook loads an anonymised snapshot directly from Hugging Face and runs:

- Stepwise p-adic linear regression (UMLLR-style)
- Decision tree (bag-of-tags)
- Small MLP (bag-of-tags)
- Zubarev simulated annealing p-adic regression

The p-adic baselines are implemented inline (pure Python) so the notebook runs
without installing any project-specific package.

Expected dataset layout in the HF repo:

- `paper/snapshot.json`, `paper/tags.jsonl.gz`, `paper/products-*.jsonl.gz`
- `latest/snapshot.json`, `latest/tags.jsonl.gz`, `latest/products-*.jsonl.gz`


In [ ]:
import json
import math
import os
import random
import re
import urllib.parse
import urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

_PATH_SPLIT_RE = re.compile(r"[>/|]+")
_SEGMENT_NUMBER_RE = re.compile(r"-?\d+")


def parse_taxonomy_path(path: Any) -> Tuple[str, ...]:
    # Normalise a taxonomy path into a tuple of segments (root-to-leaf).

    if path is None:
        return ()

    if isinstance(path, str):
        stripped = path.strip()
        if not stripped:
            return ()
        parts = [part.strip() for part in _PATH_SPLIT_RE.split(stripped) if part.strip()]
        if parts:
            return tuple(parts)
        return (stripped,)

    if isinstance(path, Sequence) and not isinstance(path, (bytes, bytearray)):
        parts = [str(part).strip() for part in path if str(part).strip()]
        return tuple(parts)

    text = str(path).strip()
    return (text,) if text else ()


def parse_taxonomy_digits(path_value: str | None) -> Tuple[int, ...]:
    # Extract numeric digits from taxonomy path segments.

    if not path_value:
        return ()

    digits: List[int] = []
    for segment in parse_taxonomy_path(path_value):
        segment = segment.strip()
        if not segment:
            continue
        try:
            digits.append(int(segment))
            continue
        except ValueError:
            matches = _SEGMENT_NUMBER_RE.findall(segment)
            if matches:
                digits.extend(int(match) for match in matches)
                continue
    return tuple(digits)


def encode_path(digits: Sequence[int], base: int) -> int:
    value = 0
    for power, digit in enumerate(digits):
        value += digit * (base ** power)
    return value


def is_prime(value: int) -> bool:
    if value < 2:
        return False
    if value in {2, 3}:
        return True
    if value % 2 == 0:
        return False
    limit = int(math.isqrt(value)) + 1
    for factor in range(3, limit, 2):
        if value % factor == 0:
            return False
    return True


def next_prime(min_value: int) -> int:
    candidate = max(2, min_value + 1)
    while True:
        if is_prime(candidate):
            return candidate
        candidate += 1


def p_adic_distance(a: int, b: int, base: int) -> float:
    if a == b:
        return 0.0

    diff = abs(a - b)
    valuation = 0
    while diff % base == 0:
        diff //= base
        valuation += 1
    return base ** (-valuation)


@dataclass(frozen=True)
class ProductRecord:
    product_id: int
    tags: List[str]
    encoded_path: int
    cv_fold: int


@dataclass(frozen=True)
class BattleRecord:
    winner_tag: str
    loser_tag: str
    cv_fold: int | None


@dataclass(frozen=True)
class TagCoefficient:
    tag: str
    coefficient: int
    sequence: int


@dataclass(frozen=True)
class Prediction:
    product_id: int
    true_value: int
    predicted_value: int
    loss: float


@dataclass(frozen=True)
class UMLLRFoldResult:
    cv_fold: int
    coefficients: List[TagCoefficient]
    predictions: List[Prediction]
    loss: float
    default_prediction: int


def _tag_order(
    battles: Sequence[BattleRecord],
    holdout_fold: int,
    training_tags: Iterable[str],
) -> List[str]:
    wins: Dict[str, int] = {}
    losses: Dict[str, int] = {}

    for battle in battles:
        if battle.cv_fold == holdout_fold:
            continue
        wins[battle.winner_tag] = wins.get(battle.winner_tag, 0) + 1
        losses[battle.loser_tag] = losses.get(battle.loser_tag, 0) + 1

    ordered_tags = list({tag for tag in training_tags})
    for tag in ordered_tags:
        wins.setdefault(tag, 0)
        losses.setdefault(tag, 0)

    ordered_tags.sort(key=lambda tag: (-wins[tag], losses[tag], tag))
    return ordered_tags


def _select_coefficient(values: Sequence[int], base: int) -> int:
    unique_values = sorted(set(values))
    best_value = unique_values[0]
    best_loss = math.inf

    for candidate in unique_values:
        total_distance = sum(p_adic_distance(candidate, value, base) for value in values)
        if total_distance < best_loss or (
            math.isclose(total_distance, best_loss) and candidate < best_value
        ):
            best_loss = total_distance
            best_value = candidate

    return best_value


def _select_default_prediction(
    no_tag_values: Sequence[int],
    candidate_values: Sequence[int],
    base: int,
) -> int:
    if not no_tag_values:
        if candidate_values:
            return Counter(candidate_values).most_common(1)[0][0]
        return 0

    unique_candidates = sorted(set(candidate_values)) if candidate_values else [0]
    best_value = unique_candidates[0]
    best_loss = float("inf")

    for candidate in unique_candidates:
        total_loss = sum(p_adic_distance(candidate, value, base) for value in no_tag_values)
        if total_loss < best_loss or (total_loss == best_loss and candidate < best_value):
            best_loss = total_loss
            best_value = candidate

    return best_value


def umllr_run_fold(
    fold: int,
    records: Sequence[ProductRecord],
    battles: Sequence[BattleRecord],
    base: int,
) -> UMLLRFoldResult:
    training = [record for record in records if record.cv_fold != fold]
    testing = [record for record in records if record.cv_fold == fold]

    product_residuals: Dict[int, int] = {
        record.product_id: record.encoded_path for record in training
    }
    tag_to_products: Dict[str, List[int]] = {}
    for record in training:
        for tag in record.tags:
            tag_to_products.setdefault(tag, []).append(record.product_id)

    tag_order = _tag_order(battles, fold, tag_to_products.keys())

    coefficients: List[TagCoefficient] = []
    for sequence, tag in enumerate(tag_order):
        product_ids = tag_to_products.get(tag, [])
        values = [product_residuals[pid] for pid in product_ids]

        if values:
            coefficient = _select_coefficient(values, base)
            for pid in product_ids:
                product_residuals[pid] -= coefficient
        else:
            coefficient = 0

        coefficients.append(
            TagCoefficient(tag=tag, coefficient=coefficient, sequence=sequence)
        )

    coefficient_lookup = {entry.tag: entry.coefficient for entry in coefficients}

    no_tag_training_values = [
        record.encoded_path
        for record in training
        if sum(coefficient_lookup.get(tag, 0) for tag in record.tags) == 0
    ]
    all_training_values = [record.encoded_path for record in training]

    default_prediction = _select_default_prediction(
        no_tag_training_values, all_training_values, base
    )

    predictions: List[Prediction] = []
    total_loss = 0.0
    for record in testing:
        predicted = sum(coefficient_lookup.get(tag, 0) for tag in record.tags)
        if predicted == 0:
            predicted = default_prediction
        loss = p_adic_distance(predicted, record.encoded_path, base)
        total_loss += loss
        predictions.append(
            Prediction(
                product_id=record.product_id,
                true_value=record.encoded_path,
                predicted_value=predicted,
                loss=loss,
            )
        )

    return UMLLRFoldResult(
        cv_fold=fold,
        coefficients=coefficients,
        predictions=predictions,
        loss=total_loss,
        default_prediction=default_prediction,
    )


# Zubarev simulated annealing regression (p-adic)

@dataclass(frozen=True)
class ZubarevFoldResult:
    cv_fold: int
    coefficients: List[TagCoefficient]
    predictions: List[Prediction]
    loss: float
    default_prediction: int
    iterations_used: int


def _binomial(n: int, k: int) -> int:
    if k < 0:
        return 0
    if k == 0:
        return 1
    if k > abs(n) and n >= 0:
        return 0

    result = 1
    for i in range(k):
        result = result * (n - i) // (i + 1)
    return result


def _mahler_predict(s: int, weights: Sequence[int]) -> int:
    result = 0
    for k, w in enumerate(weights):
        result += w * _binomial(s, k)
    return result


def _compute_loss(
    records: Sequence[ProductRecord],
    coefficients: Mapping[str, int],
    mahler_weights: Sequence[int],
    default_prediction: int,
    base: int,
) -> float:
    total_loss = 0.0

    for record in records:
        s = sum(coefficients.get(tag, 0) for tag in record.tags)
        predicted = _mahler_predict(s, mahler_weights) if mahler_weights else s

        if predicted == 0 and not any(coefficients.get(tag, 0) != 0 for tag in record.tags):
            predicted = default_prediction

        total_loss += p_adic_distance(predicted, record.encoded_path, base)

    return total_loss


def _initialize_coefficients_umllr_style(
    training: Sequence[ProductRecord],
    battles: Sequence[BattleRecord],
    holdout_fold: int,
    base: int,
) -> Dict[str, int]:
    tag_to_products: Dict[str, List[int]] = {}
    for record in training:
        for tag in record.tags:
            tag_to_products.setdefault(tag, []).append(record.product_id)

    tag_order = _tag_order(battles, holdout_fold, set(tag_to_products.keys()))

    product_residuals: Dict[int, int] = {
        record.product_id: record.encoded_path for record in training
    }
    coefficients: Dict[str, int] = {}

    for tag in tag_order:
        product_ids = tag_to_products.get(tag, [])
        values = [product_residuals[pid] for pid in product_ids]

        if values:
            coefficient = _select_coefficient(values, base)
            coefficients[tag] = coefficient
            for pid in product_ids:
                product_residuals[pid] -= coefficient
        else:
            coefficients[tag] = 0

    return coefficients


def _stochastic_optimize(
    training: Sequence[ProductRecord],
    validation: Sequence[ProductRecord],
    initial_coefficients: Dict[str, int],
    base: int,
    *,
    mahler_degree: int = 0,
    max_iterations: int = 10000,
    initial_temperature: float = 1.0,
    cooling_rate: float = 0.9995,
    min_temperature: float = 0.001,
    perturbation_scale: int = 1000,
    seed: int | None = None,
) -> Tuple[Dict[str, int], List[int], float, int]:
    if seed is not None:
        random.seed(seed)

    coefficients = dict(initial_coefficients)
    tags = list(coefficients.keys())

    mahler_weights = [0] + [1] + [0] * (mahler_degree - 1) if mahler_degree > 0 else []

    all_values = sorted({r.encoded_path for r in training})
    default_prediction = all_values[0] if all_values else 0

    current_loss = _compute_loss(training, coefficients, mahler_weights, default_prediction, base)
    best_coefficients = dict(coefficients)
    best_mahler = list(mahler_weights)
    best_loss = current_loss

    temperature = initial_temperature
    iteration = 0

    while iteration < max_iterations and temperature > min_temperature:
        if not tags:
            break

        tag = random.choice(tags)
        old_value = coefficients.get(tag, 0)

        if random.random() < 0.5:
            power = random.randint(0, 5)
            sign = random.choice([-1, 1])
            delta = sign * (base ** power)
        else:
            delta = random.randint(-perturbation_scale, perturbation_scale)

        coefficients[tag] = old_value + delta
        new_loss = _compute_loss(training, coefficients, mahler_weights, default_prediction, base)

        accepted = False
        if new_loss < current_loss:
            current_loss = new_loss
            accepted = True
            if new_loss < best_loss:
                best_loss = new_loss
                best_coefficients = dict(coefficients)
                best_mahler = list(mahler_weights)
        elif temperature > 0:
            delta_loss = new_loss - current_loss
            try:
                acceptance_prob = math.exp(-delta_loss / temperature)
            except OverflowError:
                acceptance_prob = 0.0
            if random.random() < acceptance_prob:
                current_loss = new_loss
                accepted = True

        if not accepted:
            coefficients[tag] = old_value

        temperature *= cooling_rate
        iteration += 1

    return best_coefficients, best_mahler, best_loss, iteration


def zubarev_run_fold(
    fold: int,
    records: Sequence[ProductRecord],
    battles: Sequence[BattleRecord],
    base: int,
    *,
    mahler_degree: int = 0,
    max_iterations: int = 10000,
    seed: int | None = None,
    validation_fraction: float = 0.2,
    initialization_method: str = "umllr",
) -> ZubarevFoldResult:
    all_training = [r for r in records if r.cv_fold != fold]
    testing = [r for r in records if r.cv_fold == fold]

    fold_seed = seed + fold if seed is not None else None
    if fold_seed is not None:
        random.seed(fold_seed)

    shuffled = list(all_training)
    random.shuffle(shuffled)
    val_size = int(len(shuffled) * validation_fraction)
    validation = shuffled[:val_size]
    training = shuffled[val_size:]

    if initialization_method == "umllr":
        initial_coefficients = _initialize_coefficients_umllr_style(all_training, battles, fold, base)
    elif initialization_method == "zeros":
        all_tags: set[str] = set()
        for record in all_training:
            all_tags.update(record.tags)
        initial_coefficients = {tag: 0 for tag in all_tags}
    else:
        raise ValueError(f"Unknown initialization_method: {initialization_method}")

    optimized_coefficients, mahler_weights, train_loss, iterations_used = _stochastic_optimize(
        training,
        validation,
        initial_coefficients,
        base,
        mahler_degree=mahler_degree,
        max_iterations=max_iterations,
        seed=fold_seed,
    )

    all_training_values = [r.encoded_path for r in training]
    no_tag_training_values = [
        r.encoded_path
        for r in training
        if sum(optimized_coefficients.get(tag, 0) for tag in r.tags) == 0
    ]
    default_prediction = _select_default_prediction(no_tag_training_values, all_training_values, base)

    predictions: List[Prediction] = []
    total_loss = 0.0

    for record in testing:
        s = sum(optimized_coefficients.get(tag, 0) for tag in record.tags)
        predicted = _mahler_predict(s, mahler_weights) if mahler_weights else s
        if predicted == 0 and not any(optimized_coefficients.get(tag, 0) != 0 for tag in record.tags):
            predicted = default_prediction

        loss = p_adic_distance(predicted, record.encoded_path, base)
        total_loss += loss
        predictions.append(
            Prediction(
                product_id=record.product_id,
                true_value=record.encoded_path,
                predicted_value=predicted,
                loss=loss,
            )
        )

    ordered_tags = _tag_order(battles, fold, set(optimized_coefficients.keys()))
    coefficients = [
        TagCoefficient(tag=tag, coefficient=int(optimized_coefficients.get(tag, 0)), sequence=i)
        for i, tag in enumerate(ordered_tags)
    ]

    return ZubarevFoldResult(
        cv_fold=fold,
        coefficients=coefficients,
        predictions=predictions,
        loss=total_loss,
        default_prediction=default_prediction,
        iterations_used=iterations_used,
    )


In [ ]:
DATASET_ID = os.getenv("PRODUCT_TAXONOMY_BENCH_DATASET_ID", "gregb/product-taxonomy-bench")
REVISION = os.getenv("PRODUCT_TAXONOMY_BENCH_REVISION", "main")
SNAPSHOT = os.getenv("PRODUCT_TAXONOMY_BENCH_SNAPSHOT", "latest")
HF_TOKEN = os.getenv("HF_TOKEN")
MAX_PRODUCTS = int(os.getenv("PRODUCT_TAXONOMY_BENCH_MAX_PRODUCTS", "0")) or None


def _hf_headers() -> dict[str, str]:
    if HF_TOKEN:
        return {"Authorization": f"Bearer {HF_TOKEN}"}
    return {}


def hf_api_json(path: str) -> dict:
    url = f"https://huggingface.co/api/{path.lstrip('/')}"
    request = urllib.request.Request(url, headers=_hf_headers())
    with urllib.request.urlopen(request) as response:
        return json.load(response)


def hf_resolve_url(path: str) -> str:
    dataset_slug = DATASET_ID
    revision_slug = urllib.parse.quote(REVISION, safe="")
    path_slug = urllib.parse.quote(path.lstrip("/"), safe="/")
    return f"https://huggingface.co/datasets/{dataset_slug}/resolve/{revision_slug}/{path_slug}"


def load_json(path: str) -> dict:
    request = urllib.request.Request(hf_resolve_url(path), headers=_hf_headers())
    with urllib.request.urlopen(request) as response:
        return json.load(response)


def load_jsonl(path: str) -> pd.DataFrame:
    return pd.read_json(hf_resolve_url(path), lines=True, compression="infer")


snapshot_prefix = SNAPSHOT.strip("/") + "/"
dataset_meta = hf_api_json(f"datasets/{DATASET_ID}")
all_paths = [entry.get("rfilename", "") for entry in dataset_meta.get("siblings", [])]
snapshot_paths = [path for path in all_paths if path.startswith(snapshot_prefix)]
if not snapshot_paths:
    raise ValueError(
        f"Snapshot folder {SNAPSHOT!r} not found in dataset {DATASET_ID!r}. "
        f"Available top-level folders: {sorted({p.split('/', 1)[0] for p in all_paths if p})}"
    )

snapshot_json_path = snapshot_prefix + "snapshot.json"
if snapshot_json_path not in snapshot_paths:
    raise ValueError(f"Missing {snapshot_json_path!r} in dataset {DATASET_ID!r}")

if snapshot_prefix + "tags.jsonl.gz" in snapshot_paths:
    tags_path = snapshot_prefix + "tags.jsonl.gz"
elif snapshot_prefix + "tags.jsonl" in snapshot_paths:
    tags_path = snapshot_prefix + "tags.jsonl"
else:
    raise ValueError(f"Missing tags JSONL file in snapshot {SNAPSHOT!r}")

products_pattern = re.compile(
    rf"^{re.escape(snapshot_prefix)}products-\d+\.jsonl(?:\.gz)?$"
)
product_paths = sorted(path for path in snapshot_paths if products_pattern.match(path))
if not product_paths:
    raise ValueError(f"No products JSONL shards found for snapshot {SNAPSHOT!r}")

snapshot_metadata = load_json(snapshot_json_path)
tags = load_jsonl(tags_path).sort_values("tag_rank").reset_index(drop=True)
product_frames = [load_jsonl(path) for path in product_paths]
products_raw = pd.concat(product_frames, ignore_index=True)

if MAX_PRODUCTS:
    products_raw = (
        products_raw.sort_values("product_id_hash").head(MAX_PRODUCTS).reset_index(drop=True)
    )

products = products_raw[
    [
        "product_id_hash",
        "taxonomy_id",
        "taxonomy_path",
        "taxonomy_name",
        "cv_fold",
        "tag_count",
        "title_part_count",
    ]
].copy()

product_tags = (
    products_raw[["product_id_hash", "tag_features"]]
    .explode("tag_features", ignore_index=True)
    .dropna(subset=["tag_features"])
)
if not product_tags.empty:
    expanded = pd.json_normalize(product_tags["tag_features"])
    product_tags = pd.concat(
        [product_tags.drop(columns=["tag_features"]), expanded],
        axis=1,
    )
else:
    product_tags = pd.DataFrame(
        columns=["product_id_hash", "tag_id", "in_title", "title_part", "title_position"]
    )

product_tags = product_tags.dropna(subset=["tag_id"]).copy()
product_tags["tag_id"] = product_tags["tag_id"].astype(str)
product_tags["in_title"] = product_tags["in_title"].fillna(False).astype(bool)
product_tags["title_part"] = pd.to_numeric(product_tags["title_part"], errors="coerce")
product_tags["title_position"] = pd.to_numeric(
    product_tags["title_position"], errors="coerce"
)

title_tags = product_tags[
    product_tags["in_title"] & product_tags["title_position"].notna()
][["product_id_hash", "title_part", "tag_id", "title_position"]].copy()
title_tags["title_part"] = title_tags["title_part"].fillna(0).astype(int)
title_tags["title_position"] = title_tags["title_position"].astype(int)

print("dataset", DATASET_ID, "revision", REVISION)
print("snapshot folder", SNAPSHOT)
print("snapshot name", snapshot_metadata.get("snapshot_name"))
print("products", len(products), "tags", len(tags), "shards", len(product_paths))
products.head()


In [ ]:
# Build bag-of-tags sparse matrix X and label vector y

products = products.dropna(subset=["cv_fold"]).copy()
products["cv_fold"] = products["cv_fold"].astype(int)
products = products.sort_values("product_id_hash").reset_index(drop=True)

tag_to_col = {tag_id: i for i, tag_id in enumerate(tags["tag_id"].tolist())}
product_to_row = {pid: i for i, pid in enumerate(products["product_id_hash"].tolist())}

rows = []
cols = []
data = []

for pid, tag_id in product_tags[["product_id_hash", "tag_id"]].itertuples(index=False, name=None):
    row_idx = product_to_row.get(pid)
    col_idx = tag_to_col.get(tag_id)
    if row_idx is None or col_idx is None:
        continue
    rows.append(row_idx)
    cols.append(col_idx)
    data.append(1.0)

X = sparse.csr_matrix(
    (data, (rows, cols)),
    shape=(len(products), len(tags)),
    dtype=np.float32,
)

y = products["taxonomy_id"].to_numpy(dtype=object)

print("products", X.shape[0])
print("tags", X.shape[1])
print("taxonomies", len(set(y)))

In [ ]:
# Build taxonomy encodings + p-adic base for evaluation

taxonomy_paths = (
    products[["taxonomy_id", "taxonomy_path"]]
    .drop_duplicates(subset=["taxonomy_id"])
    .itertuples(index=False, name=None)
)

taxonomy_digits = {}
max_digit = 0
for taxonomy_id, taxonomy_path in taxonomy_paths:
    digits = parse_taxonomy_digits(taxonomy_path)
    taxonomy_digits[taxonomy_id] = digits
    if digits:
        max_digit = max(max_digit, max(digits))

base = next_prime(max_digit)
taxonomy_encoded = {
    taxonomy_id: encode_path(digits, base)
    for taxonomy_id, digits in taxonomy_digits.items()
}


def mean_padic_loss(y_true, y_pred) -> float:
    losses = []
    for t, p in zip(y_true, y_pred):
        losses.append(p_adic_distance(taxonomy_encoded[t], taxonomy_encoded[p], base))
    return float(np.mean(losses)) if losses else 0.0


print("prime base", base)
print("max digit", max_digit)


In [ ]:
# Decision tree + small MLP baselines (5-fold CV using precomputed cv_fold)

from sklearn.dummy import DummyClassifier

folds = sorted(products["cv_fold"].unique().tolist())
mlp_max_iter = int(os.getenv("PRODUCT_TAXONOMY_BENCH_MLP_MAX_ITER", "40"))


def model_parameter_count(model) -> float:
    if hasattr(model, "tree_"):
        n_classes = model.n_classes_
        if isinstance(n_classes, np.ndarray):
            n_classes = int(np.max(n_classes))
        else:
            n_classes = int(n_classes)
        return float(model.tree_.node_count * math.log2(max(n_classes, 2)))

    if hasattr(model, "coefs_") and hasattr(model, "intercepts_"):
        return float(
            sum(arr.size for arr in model.coefs_) + sum(arr.size for arr in model.intercepts_)
        )

    return 0.0


def eval_cv(make_model, param_counter=model_parameter_count):
    fold_losses = []
    fold_acc = []
    fold_params = []

    for fold in folds:
        train_mask = products["cv_fold"].to_numpy() != fold
        test_mask = products["cv_fold"].to_numpy() == fold

        model = make_model()
        model.fit(X[train_mask], y[train_mask])
        y_pred = model.predict(X[test_mask])

        fold_losses.append(mean_padic_loss(y[test_mask], y_pred))
        fold_acc.append(float(np.mean(y_pred == y[test_mask])))
        fold_params.append(float(param_counter(model)))

    return (
        float(np.mean(fold_losses)),
        fold_losses,
        float(np.mean(fold_acc)),
        fold_acc,
        float(np.mean(fold_params)),
        fold_params,
    )


dt_mean_loss, dt_losses, dt_mean_acc, dt_acc, dt_mean_params, dt_params = eval_cv(
    lambda: DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=42)
)

mlp_mean_loss, mlp_losses, mlp_mean_acc, mlp_acc, mlp_mean_params, mlp_params = eval_cv(
    lambda: MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        alpha=1e-4,
        batch_size=1024,
        max_iter=mlp_max_iter,
        random_state=42,
    )
)

dummy_mean_loss, dummy_losses, dummy_mean_acc, dummy_acc, dummy_mean_params, dummy_params = eval_cv(
    lambda: DummyClassifier(strategy="most_frequent"),
    param_counter=lambda _model: 1.0,
)

print("Decision tree mean p-adic loss", dt_mean_loss, "mean acc", dt_mean_acc, "mean params", dt_mean_params)
print("MLP mean p-adic loss", mlp_mean_loss, "mean acc", mlp_mean_acc, "mean params", mlp_mean_params)
print("Dummy mean p-adic loss", dummy_mean_loss, "mean acc", dummy_mean_acc, "mean params", dummy_mean_params)


In [ ]:
# Build UMLLR/Zubarev records + battles from the anonymised snapshot

tags_by_product = defaultdict(list)
for pid, tag_id in product_tags[["product_id_hash", "tag_id"]].itertuples(index=False, name=None):
    tags_by_product[pid].append(tag_id)

fold_by_product = dict(products[["product_id_hash", "cv_fold"]].itertuples(index=False, name=None))

records = []
for idx, (pid, taxonomy_id, cv_fold) in enumerate(
    products[["product_id_hash", "taxonomy_id", "cv_fold"]].itertuples(index=False, name=None)
):
    records.append(
        ProductRecord(
            product_id=idx,
            tags=tags_by_product.get(pid, []),
            encoded_path=taxonomy_encoded[taxonomy_id],
            cv_fold=int(cv_fold),
        )
    )


# Battles are derived from tags that overlap title text. Each title part becomes its own arena.
# ordered is ascending by title_position; later tags win against earlier ones.

battles = []
group_cols = ["product_id_hash", "title_part"]
for (pid, part), group in title_tags.groupby(group_cols):
    fold = fold_by_product.get(pid)
    if fold is None:
        continue
    ordered = group.sort_values("title_position")[["tag_id", "title_position"]].to_numpy()
    for i in range(len(ordered)):
        for j in range(i + 1, len(ordered)):
            loser = str(ordered[i][0])
            winner = str(ordered[j][0])
            if loser == winner:
                continue
            battles.append(
                BattleRecord(winner_tag=winner, loser_tag=loser, cv_fold=int(fold))
            )

print("records", len(records))
print("battles", len(battles))


In [ ]:
# Stepwise p-adic linear regression (UMLLR-style)

umllr_fold_losses = []
umllr_nonzero_params = []
for fold in folds:
    result = umllr_run_fold(fold, records, battles, base)
    testing_count = sum(1 for r in records if r.cv_fold == fold)
    mean_loss = (result.loss / testing_count) if testing_count else 0.0
    umllr_fold_losses.append(mean_loss)
    umllr_nonzero_params.append(sum(1 for c in result.coefficients if c.coefficient != 0))

print("UMLLR mean p-adic loss", float(np.mean(umllr_fold_losses)))
print("UMLLR nonzero params (per fold)", umllr_nonzero_params)


In [ ]:
# Zubarev simulated annealing p-adic regression
# NOTE: this can be slow; reduce max_iterations for quick checks.

zubarev_max_iterations = int(
    os.getenv("PRODUCT_TAXONOMY_BENCH_ZUBAREV_MAX_ITERATIONS", "2000")
)

zub_fold_losses = []
zub_nonzero_params = []
for fold in folds:
    result = zubarev_run_fold(
        fold,
        records,
        battles,
        base,
        mahler_degree=0,
        max_iterations=zubarev_max_iterations,
        seed=42,
        initialization_method="umllr",
    )
    testing_count = sum(1 for r in records if r.cv_fold == fold)
    mean_loss = (result.loss / testing_count) if testing_count else 0.0
    zub_fold_losses.append(mean_loss)
    zub_nonzero_params.append(sum(1 for c in result.coefficients if c.coefficient != 0))

print("Zubarev mean p-adic loss", float(np.mean(zub_fold_losses)))
print("Zubarev nonzero params (per fold)", zub_nonzero_params)


In [ ]:
# Final model comparison + parsimoniousness metric

import matplotlib.pyplot as plt

PARSIMONY_BASELINE_SLOPE = -2.0
PARSIMONY_BASELINE_INTERCEPT = -0.1
EPS = 1e-12

model_results = pd.DataFrame(
    [
        {
            "model": "Dummy",
            "params": float(dummy_mean_params),
            "mean_padic_loss": float(dummy_mean_loss),
            "mean_accuracy": float(dummy_mean_acc),
        },
        {
            "model": "UMLLR",
            "params": float(np.mean(umllr_nonzero_params)),
            "mean_padic_loss": float(np.mean(umllr_fold_losses)),
            "mean_accuracy": np.nan,
        },
        {
            "model": "Decision Tree",
            "params": float(dt_mean_params),
            "mean_padic_loss": float(dt_mean_loss),
            "mean_accuracy": float(dt_mean_acc),
        },
        {
            "model": "MLP",
            "params": float(mlp_mean_params),
            "mean_padic_loss": float(mlp_mean_loss),
            "mean_accuracy": float(mlp_mean_acc),
        },
        {
            "model": "Zubarev (UMLLR init)",
            "params": float(np.mean(zub_nonzero_params)),
            "mean_padic_loss": float(np.mean(zub_fold_losses)),
            "mean_accuracy": np.nan,
        },
    ]
)

model_results["log10_params"] = np.log10(model_results["params"].clip(lower=1.0))
model_results["log10_loss"] = np.log10(model_results["mean_padic_loss"].clip(lower=EPS))
model_results["baseline_log10_loss"] = (
    PARSIMONY_BASELINE_SLOPE * model_results["log10_params"] + PARSIMONY_BASELINE_INTERCEPT
)
model_results["parsimony_score"] = (
    model_results["baseline_log10_loss"] - model_results["log10_loss"]
)
model_results["better_than_baseline"] = model_results["parsimony_score"] > 0

model_results = model_results.sort_values("parsimony_score", ascending=False).reset_index(drop=True)

fig, (ax_scatter, ax_bar) = plt.subplots(1, 2, figsize=(14, 5))

ax_scatter.scatter(
    model_results["log10_params"],
    model_results["log10_loss"],
    s=100,
    color="#0b6ce3",
)
for row in model_results.itertuples(index=False):
    ax_scatter.annotate(row.model, (row.log10_params, row.log10_loss), textcoords="offset points", xytext=(6, 4), fontsize=9)

x_min = model_results["log10_params"].min() - 0.2
x_max = model_results["log10_params"].max() + 0.2
x_line = np.linspace(x_min, x_max, 200)
y_line = PARSIMONY_BASELINE_SLOPE * x_line + PARSIMONY_BASELINE_INTERCEPT
ax_scatter.plot(x_line, y_line, "--", color="#ef4444", linewidth=2, label="Baseline: log10(loss) = -2·log10(params) - 0.1")
ax_scatter.set_xlabel("log10(params)")
ax_scatter.set_ylabel("log10(p-adic loss)")
ax_scatter.set_title("Model size vs p-adic loss")
ax_scatter.grid(alpha=0.3)
ax_scatter.legend(fontsize=8)

bar_colors = ["#16a34a" if value > 0 else "#dc2626" for value in model_results["parsimony_score"]]
ax_bar.bar(model_results["model"], model_results["parsimony_score"], color=bar_colors)
ax_bar.axhline(0, color="#334155", linewidth=1)
ax_bar.set_ylabel("Parsimony score (baseline - observed log10 loss)")
ax_bar.set_title("Parsimoniousness by model")
ax_bar.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

results_table = model_results[
    [
        "model",
        "params",
        "mean_padic_loss",
        "mean_accuracy",
        "log10_params",
        "log10_loss",
        "baseline_log10_loss",
        "parsimony_score",
        "better_than_baseline",
    ]
].copy()

results_table["params"] = results_table["params"].round(1)
results_table["mean_padic_loss"] = results_table["mean_padic_loss"].round(6)
results_table["mean_accuracy"] = results_table["mean_accuracy"].round(4)
results_table["log10_params"] = results_table["log10_params"].round(4)
results_table["log10_loss"] = results_table["log10_loss"].round(4)
results_table["baseline_log10_loss"] = results_table["baseline_log10_loss"].round(4)
results_table["parsimony_score"] = results_table["parsimony_score"].round(4)

print("Parsimony baseline: log10(loss) = -2 * log10(params) - 0.1")
results_table
